# EDA — 학교명 댓글 데이터셋 (`data/raw/dataset.csv`)

In [ ]:
import sys
from pathlib import Path

# 커널이 어디서 시작되든(repo 루트든 src/ 안이든) 항상 동작하도록
# data/, src/ 폴더를 둘 다 가진 위치를 repo 루트로 잡는다.
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src" / "data") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src" / "data"))

REPO_ROOT

In [ ]:
import pandas as pd

df = pd.read_csv(REPO_ROOT / "data" / "raw" / "dataset.csv", encoding="utf-8")
df.shape

## 결측치 / 중복

In [22]:
df.isna().sum()

comment_id    0
comment       0
dtype: int64

In [23]:
print("comment_id 중복:", df["comment_id"].duplicated().sum())
print("comment 중복:", df["comment"].duplicated().sum())

comment_id 중복: 0
comment 중복: 226


In [24]:
dup_counts = df["comment"].value_counts()
dup_counts = dup_counts[dup_counts > 1]
print("중복된 고유 comment 개수:", len(dup_counts))
dup_counts

중복된 고유 comment 개수: 173


comment
우리 반/동아리 대표로 반포중 신청합니다!        6
치킨 오면 인증샷 바로 올릴게요 잠원초          5
서울고 축제 준비팀입니다 치킨이면 밤샘 가능       4
우리 반/동아리 대표로 서울고 신청합니다!        4
서초중도 참여 완료!                    4
                              ..
배민 이벤트 참여합니다! 잠실중로 보내주세요       2
비 오는 날엔 치킨이죠 인하대부속고 부탁드립니다     2
인하대부속중ㅋㅋ 치킨 기다립니다              2
서초초도 참여 완료!                    2
비 오는 날엔 치킨이죠 이화여대부속중 부탁드립니다    2
Name: count, Length: 173, dtype: int64

이거를 오류라고 봐야하나? (친구들한테 부탁해서 복붙한 메세지인거 아닐까?) 
=> 오류라 판단 불가  

In [25]:
df.head(10)

,comment_id,comment
0,C0001,동국대학교
1,C0002,이번엔 중동고 차례입니다 🍗
2,C0003,우리 반/동아리 대표로 서초초 신청합니다!
3,C0004,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요
4,C0005,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청!
5,C0006,학식 말고 치킨 먹고 싶어요 인하대학교
6,C0007,서울공업고.. 오늘만 기다렸어요
7,C0008,서강대 학생들 모여라
8,C0009,동아리방에서 기다릴게요 서초중
9,C0010,대치중 학생입니다 대치중 뽑아주세요


< 다음 스텝 >
- 이모티콘, ! 등등 다 삭제 
- 조사 및 동사 삭제 ( commend_id는 남겨둘 것 count != 1인 경우엔 확인 필요)

< 확인 >
- 건국대 -> 건대 줄이는 경우 발견 => 표준적인 대학교 이름이 아닌 경우 존재하므로 원래 있는 데이터를 사용하는 것보단 "대", "중", "초", "학교"로 끝나는 애들로 판단해야함
    - 이때 학교가 아닌 경우는? 
- 한 문장에 여러 학교 나오는 경우 발견 => count해서 문맥 파악 필요 
    - 당장은 판단 어려울듯 

< 의문 >
- 이화 여대 이렇게 띄어쓰기 한 사람 있을까?